# Tutorial: Magnetic Pattern Definition

This notebook isolates the magnetic-domain pattern controls.

The scalar magnetic pattern represents the out-of-plane magnetization `m_z` and should usually live in `[-1, 1]`. Later, the scalar pattern is expanded to a 3-D vector field `(mx, my, mz)` and repeated through the sample layers.


In [ ]:
import sys
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

# Make the notebook runnable from a fresh clone without requiring an editable install.
repo_root = Path.cwd()
if (repo_root / "src").exists() and str(repo_root / "src") not in sys.path:
    sys.path.insert(0, str(repo_root / "src"))

try:
    # Use the interactive widget backend when it is available in JupyterLab.
    %matplotlib widget
except Exception:
    # Plain scripts and some notebook renderers do not understand IPython magics.
    pass

plt.rcParams["figure.constrained_layout.use"] = True

from scattering_calculator.simulation_pipelines import simulation_configuration as sim
from scattering_calculator.sample_generator import pattern_generator


## 1. Common grid

The configuration accepts physical length scales in metres and converts them to pixels through `real_space_pixel_size`.


In [ ]:
shape = (384, 384)
real_space_pixel_size = 4e-9


## 2. Binary labyrinth pattern

This is a good default for didactic magnetic domains because it creates two-domain contrast with an interpretable stripe width.


In [ ]:
labyrinth_config = sim.MagneticPatternConfig(
    pattern_type_method="binary_labyrinth_pattern",
    shape=shape,
    real_space_pixel_size=real_space_pixel_size,
    pattern_config={
        "stripe_width": 55e-9,
        "sigma": 4e-9,
        "domain_conversion": "soft",
        "softness": 1.2,
        "n_steps": 80,
        "seed": 2,
        "use_gpu": False,
    },
)
labyrinth_config.create_pattern()
labyrinth_config.plot_pattern()


## 3. Wavy stripes and saturated state

These two cells show two extremes: structured domains and uniform magnetization.


In [ ]:
wavy_config = sim.MagneticPatternConfig(
    pattern_type_method="wavy_stripe_pattern",
    shape=shape,
    real_space_pixel_size=real_space_pixel_size,
    pattern_config={
        "stripe_width": 65e-9,
        "angle_stripes": np.deg2rad(20),
        "sigma": 5e-9,
        "waviness_amplitude": 45e-9,
        "waviness_scale": 220e-9,
        "seed": 5,
    },
)
wavy_config.create_pattern()

saturated_config = sim.MagneticPatternConfig(
    pattern_type_method="saturated_pattern",
    shape=shape,
    real_space_pixel_size=real_space_pixel_size,
    pattern_config={"saturation": 1.0},
)
saturated_config.create_pattern()

fig, axes = plt.subplots(1, 2, figsize=(8, 3.5))
for ax, cfg, title in zip(axes, [wavy_config, saturated_config], ["wavy stripes", "saturated"]):
    ax.imshow(cfg.magnetic_pattern, cmap="gray", vmin=-1, vmax=1)
    ax.set_title(title)
    ax.set_axis_off()


## 4. Pattern statistics

Histograms are useful for checking whether a pattern is binary, soft, biased, or saturated.


In [ ]:
configs = [labyrinth_config, wavy_config, saturated_config]
fig, axes = plt.subplots(1, len(configs), figsize=(11, 3.2), sharey=True)
for ax, cfg in zip(axes, configs):
    values = cfg.magnetic_pattern.ravel()
    ax.hist(values, bins=60, range=(-1, 1), color="0.25")
    ax.set_title(cfg.pattern_type_method)
    ax.set_xlabel("m_z")
    ax.set_ylabel("pixels")
    print(cfg.pattern_type_method, "min/max/mean:", values.min(), values.max(), values.mean())


## 5. Convert scalar `m_z` to a 3-D magnetization vector field

The CK notebook uses mostly out-of-plane magnetization. The helper below stacks `(mx, my, mz)` and repeats it for every material layer.


In [ ]:
n_layers = 4
mz = labyrinth_config.magnetic_pattern
mx = np.zeros_like(mz)

# One simple teaching convention: put the leftover norm into my.
# For fully binary mz = +/-1 this gives my = 0.
my = np.sqrt(np.clip(1 - np.abs(mz) ** 2, 0, 1))

magnetization = pattern_generator.map_magnetization_to_3d(
    magnetic_pattern_x=mx,
    magnetic_pattern_y=my,
    magnetic_pattern_z=mz,
    nr_repeats=n_layers,
)

print("magnetization shape:", magnetization.shape)
print("last axis is (mx, my, mz)")


## 6. Visualize vector components for one layer

In [ ]:
layer = 0
fig, axes = plt.subplots(1, 3, figsize=(10, 3.2))
for i, label in enumerate(["mx", "my", "mz"]):
    im = axes[i].imshow(magnetization[layer, :, :, i], cmap="RdBu_r", vmin=-1, vmax=1)
    axes[i].set_title(label)
    axes[i].set_axis_off()
fig.colorbar(im, ax=axes, label="magnetization component")
